# DINOv2 + VPT MLP Training (ViT-B)
Run this notebook on **Kaggle** (recommended — dataset already available, free T4 GPU)  
or on **Colab Pro+** (need to download dataset manually, requires ~350 GB disk).

## 1. Check GPU

In [ ]:
!nvidia-smi

## 2. Clone repo & install dependencies

In [ ]:
!git clone https://github.com/aryangarg794/contrail-segmentation.git
%cd contrail-segmentation

In [ ]:
!pip install -q torch torchvision lightning transformers timm \
    albumentations segmentation-models-pytorch \
    hydra-core wandb dill rich tqdm
!pip install -q -e .

## 3. Dataset setup

### Option A — Kaggle notebooks (recommended)
Add the competition dataset via the Kaggle UI:  
`+ Add Data → google-research-identify-contrails-reduce-global-warming`  
It will be available at `/kaggle/input/google-research-identify-contrails-reduce-global-warming/`

### Option B — Colab (download via Kaggle API)
Upload your `kaggle.json` when prompted, then run the cell below.

In [ ]:
import os

# --- Option A: Kaggle notebook ---
DATA_ROOT = '/kaggle/input/google-research-identify-contrails-reduce-global-warming'

# --- Option B: Colab ---
# from google.colab import files
# files.upload()  # upload kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !kaggle competitions download -c google-research-identify-contrails-reduce-global-warming -p /content/data
# !unzip -q /content/data/google-research-identify-contrails-reduce-global-warming.zip -d /content/data
# DATA_ROOT = '/content/data'

TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
META_PATH = os.path.join(DATA_ROOT, 'train_metadata.json')
print('Train dir exists:', os.path.exists(TRAIN_DIR))
print('Metadata exists: ', os.path.exists(META_PATH))

## 4. Patch data paths at runtime

In [ ]:
import contrail_segmentation.data.utils as data_utils
data_utils.DATA_DIR = TRAIN_DIR
data_utils.META_PATH = META_PATH
# Reload metadata with updated path
import pandas as pd
data_utils.metadata = pd.read_json(META_PATH)

## 5. W&B login (optional — skip to disable)

In [ ]:
import wandb
wandb.login()

## 6. Build dataloaders

In [ ]:
import numpy as np
import torch
import albumentations as A
from torch.utils.data import DataLoader, Subset
from contrail_segmentation.data.dataset import ContrailDataset

SEED       = 0
BATCH_SIZE = 16
NUM_WORKERS = 2  # keep low on Kaggle/Colab

torch.manual_seed(SEED)
np.random.seed(SEED)
generator = torch.Generator().manual_seed(SEED)

train_transform = A.Compose([
    A.ShiftScaleRotate(scale_limit=0.2, rotate_limit=180, shift_limit=0.3,
                       border_mode=0, value=0, p=0.5),
    A.HorizontalFlip(),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.3, p=0.5),
])

full_dataset = ContrailDataset(mask_only=False)
indices      = np.arange(len(full_dataset))
np.random.shuffle(indices)
train_size   = int(0.8 * len(indices))
train_idx, test_idx = indices[:train_size], indices[train_size:]

train_set = ContrailDataset(mask_only=False, transform=train_transform)
test_set  = ContrailDataset(mask_only=False)

train_loader = DataLoader(Subset(train_set, train_idx), batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          pin_memory=True, num_workers=NUM_WORKERS)
test_loader  = DataLoader(Subset(test_set, test_idx),  batch_size=BATCH_SIZE,
                          pin_memory=True, num_workers=NUM_WORKERS)

print(f'Train: {len(train_idx)} samples | Val: {len(test_idx)} samples')

## 7. Build model (DINOv2 ViT-B + VPT MLP)

In [ ]:
from contrail_segmentation.models.dino_mlp import DINOv3MLP

model = DINOv3MLP(
    model_name='facebook/dinov3-vitb16-pretrain-lvd1689m',
    num_vpt=50,
    lr=1e-4,
    wd=1e-3,
    threshold=0.5,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params: {trainable:,} / {total:,}')

## 8. Train

In [ ]:
from datetime import datetime
from lightning.pytorch import Trainer
from lightning.pytorch.loggers import WandbLogger
from contrail_segmentation.train.utils import find_best_threshold

MAX_EPOCHS = 100
timestamp  = datetime.now().strftime('%d_%b_%Y__%Hh%Mm')
run_name   = f'dinov2_vitb_vpt_mlp_seed{SEED}_{timestamp}'

logger = WandbLogger(project='contrail-segmentation', name=run_name, save_dir='wandb_logs')

trainer = Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    precision='16-mixed',
    log_every_n_steps=1,
    logger=logger,
)

trainer.fit(model, train_dataloaders=train_loader)

## 9. Test

In [ ]:
best_thresh = find_best_threshold(model, test_loader)
model.threshold = best_thresh
model.mask_only = False

test_metrics = trainer.test(model, dataloaders=test_loader)
print(test_metrics)

## 10. Save checkpoint

In [ ]:
torch.save(model.state_dict(), f'dinov2_vitb_vpt_mlp_{timestamp}.pt')
print('Saved.')